# Gold Layer - Business Intelligence Marts

**Purpose**: Pre-aggregated, dashboard-ready business metrics and analytics optimized for BI tool consumption.

**Target Audience**: Business analysts, executives, dashboard consumers

**Layer Position**: Final consumption layer in the medallion architecture (Bronze → Silver → Semantic → Warehouse → **Gold**)

**Current Status**: 🚧 **Design Phase** - Warehouse layer stabilization in progress

---

## 📊 Overview

The Gold layer contains **denormalized, pre-computed fact tables** designed for:
* **Fast dashboard queries** - Pre-aggregated metrics eliminate expensive joins
* **Business-friendly structure** - Organized by business domain (salary, skills, hiring)
* **Time-series analysis** - Optimized for trending and temporal comparisons
* **Self-service analytics** - Simple schemas that business users can query directly

### Architecture Principles

* **Denormalized for Performance** - Dimensional attributes embedded in facts
* **Pre-Aggregated** - Rollups computed at ingestion time, not query time
* **Temporal Optimization** - Window functions and lag metrics for trending
* **Multiple Grain Levels** - Drill-down capability (total → sector → role → location)
* **Metadata Timestamps** - All tables include `created_at` and `batch_id` for lineage

### Current Implementation Status

**✅ Ready for Development:**
* Warehouse star schema structure complete
* Core dimensions available: company, location, sector, source, date
* Operational fact tables: `fact_pipeline_runs` (213 runs tracked)

**⚠️ Pending Upstream:**
* **Role dimension mapping**: Incomplete coverage in `semantic.inter_job_role_map` (plain text → structured IDs)
* **Salary data**: Mock data in `fact_salary` (484 records) - awaiting extraction from bronze JSON payloads
* **Skills bridge**: `bridge_job_skill` structure ready but not yet populated

**🔜 Next Steps:**
1. Complete role taxonomy mapping in intermediate layer
2. Extract actual salary data from bronze source payloads
3. Implement skill extraction and bridge table population
4. Build first gold mart: `gold_pipeline_health` (operational monitoring)

---

## 🚦 TL;DR - Current State Summary

**Gold Layer Status**: 🚧 **Not Yet Built** - Design complete, awaiting implementation

**What's Ready**:
* ✅ Warehouse layer operational (562 job posting events, 213 pipeline runs tracked)
* ✅ Core dimensions complete (company, location, sector, source, date)
* ✅ Can build operational gold tables: `gold_pipeline_health`, `gold_company_hiring`, `gold_location_trends`

**What's Blocked**:
* ⚠️ **Salary analytics**: Real salary data not extracted from bronze (mock data only)
* ⚠️ **Skills analytics**: Skill extraction not implemented (`bridge_job_skill` empty)
* ⚠️ **Role-level breakdowns**: Incomplete role taxonomy mapping (~0% coverage)

**Recommended Next Steps**:
1. **Build `gold_pipeline_health` first** - Enables DataOps monitoring, no blockers
2. Build `gold_company_hiring` - Company-level posting trends
3. Build `gold_location_trends` - Geographic hiring patterns
4. Fix upstream blockers before building salary/skills gold tables

**Read Below For**: Detailed gold table catalog, dependencies, blockers, and implementation guidance

---

## 📁 Gold Notebooks Catalog

### Implementation Priority

**Phase 1 - Operational Analytics** (✅ Ready to Build)
* `gold_pipeline_health` - DataOps monitoring (source: `fact_pipeline_runs`)
* `gold_company_hiring` - Company hiring activity (source: `fact_job_postings`)
* `gold_location_trends` - Geographic patterns (source: `fact_job_postings` + `dim_location`)

**Phase 2 - Core Business Analytics** (⚠️ Blocked - Awaiting Data)
* `gold_salary_trends` - Requires actual salary data extraction from bronze
* `gold_skill_demand` - Requires `bridge_job_skill` population

**Phase 3 - Advanced Analytics** (🔜 Future)
* Hospitality-specific marts
* Predictive hiring models
* Skills co-occurrence networks

---

## 📊 Planned Gold Tables

### **Core Analytics Marts**

#### 1. `gold_salary_trends` ⚠️ BLOCKED
**Status**: ⚠️ **BLOCKED** - Requires actual salary data extraction from bronze JSON payloads  
**Purpose**: Salary trend analytics with percentile distributions  
**Target Table**: `workspace.gold.gold_salary_trends`  
**Grain**: Date × Sector × Role × Location × Company  
**Key Metrics**: Median salaries (min/max), P25/P75/P90 percentiles, observation counts, MoM/QoQ changes  
**Rollups**: Sector-level, role-level, location-level aggregations  
**Use Cases**: Compensation benchmarking, salary trend dashboards, competitive intelligence  
**Blocker Details**: `fact_salary` currently contains mock data; silver layer lacks salary columns

#### 2. `gold_skill_demand` ⚠️ BLOCKED
**Status**: ⚠️ **BLOCKED** - Requires `bridge_job_skill` population  
**Purpose**: Skill demand trends and co-occurrence patterns  
**Target Table**: `workspace.gold.gold_skill_demand`  
**Grain**: Date × Skill × Sector × Role  
**Key Metrics**: Job postings requiring skill, co-occurrence rankings, skill trend velocity  
**Use Cases**: Skills gap analysis, curriculum planning, upskilling priorities  
**Blocker Details**: Skill extraction from job descriptions not yet implemented

#### 3. `gold_hiring_trends` ✅ READY
**Status**: ✅ **READY** - Can build from `fact_job_postings`  
**Purpose**: Job posting velocity and hiring activity indicators  
**Target Table**: `workspace.gold.gold_hiring_trends`  
**Grain**: Date × Sector × Location  
**Key Metrics**: New postings, closures, open positions, posting velocity, avg time-to-fill  
**Use Cases**: Labor market health tracking, hiring demand forecasting  
**Note**: Role-level breakdowns limited due to incomplete role mapping (484/562 jobs mapped)

#### 4. `gold_location_trends` ✅ READY
**Status**: ✅ **READY** - Can build from `fact_job_postings` + `dim_location`  
**Purpose**: Geographic hiring patterns and remote work analysis  
**Target Table**: `workspace.gold.gold_location_trends`  
**Grain**: Date × Location × Work Mode (Remote/Hybrid/Onsite)  
**Key Metrics**: Posting counts by location, remote work %, geographic concentration  
**Use Cases**: Regional labor market reports, remote work adoption tracking

#### 5. `gold_sector_overview` ⚠️ PARTIAL
**Status**: ⚠️ **PARTIAL** - Basic metrics ready, skills/salary blocked  
**Purpose**: Sector-level summary metrics and cross-sector comparisons  
**Target Table**: `workspace.gold.gold_sector_overview`  
**Grain**: Date × Sector  
**Key Metrics**: Total postings, avg salary, top skills, hiring velocity, sector growth  
**Use Cases**: Industry benchmarking, sector health reports, executive dashboards  
**Limitations**: Can compute posting counts/velocity; salary and skills blocked upstream

#### 6. `gold_company_hiring` ✅ READY
**Status**: ✅ **READY** - Can build from `fact_job_postings` + `dim_company`  
**Purpose**: Company-level hiring activity and employer brand insights  
**Target Table**: `workspace.gold.gold_company_hiring`  
**Grain**: Date × Company × Sector  
**Key Metrics**: Active postings, new postings, hiring velocity, role distribution  
**Use Cases**: Employer intelligence, competitive hiring analysis

---

### **Specialized Analytics**

#### 7. `gold_hospitality_hiring` 🔜 FUTURE
**Status**: 🔜 **FUTURE** - Planned after core marts established  
**Purpose**: Hospitality sector-specific hiring trends  
**Target Table**: `workspace.gold.gold_hospitality_hiring`  
**Grain**: Date × Hospitality Subsector × Location  
**Key Metrics**: Postings by hotel/restaurant/travel segments, seasonal patterns, recovery indicators  
**Use Cases**: Hospitality industry reports, tourism recovery tracking  
**Dependencies**: Requires hospitality subsector taxonomy in intermediate layer

#### 8. `gold_hospitality_skills` 🔜 FUTURE
**Status**: 🔜 **FUTURE** - Depends on skills extraction + hospitality taxonomy  
**Purpose**: Hospitality-specific skill demand patterns  
**Target Table**: `workspace.gold.gold_hospitality_skills`  
**Grain**: Date × Skill × Hospitality Subsector  
**Key Metrics**: In-demand hospitality skills, certification requirements, skill gaps  
**Use Cases**: Hospitality workforce planning, training program design

#### 9. `gold_hospitality_companies` 🔜 FUTURE
**Status**: 🔜 **FUTURE** - Low priority  
**Purpose**: Major hospitality employer hiring activity  
**Target Table**: `workspace.gold.gold_hospitality_companies`  
**Grain**: Date × Company  
**Key Metrics**: Postings by major brands (Marriott, Hilton, etc.), expansion indicators  
**Use Cases**: Employer-specific intelligence, industry consolidation analysis

---

### **Operational Monitoring**

#### 10. `gold_pipeline_health` ✅ READY (Priority #1)
**Status**: ✅ **READY** - Recommended first gold table to build  
**Purpose**: Data pipeline quality and operational health metrics  
**Target Table**: `workspace.gold.gold_pipeline_health`  
**Grain**: Date × Pipeline × Layer  
**Key Metrics**: Run success rate, data freshness, quality scores, processing latency  
**Use Cases**: DataOps monitoring, SLA tracking, data quality dashboards  
**Source Ready**: `fact_pipeline_runs` contains 213 pipeline executions (2 pipelines tracked)  
**Implementation Priority**: **HIGH** - Enables operational visibility before business analytics

---

## 🔧 Usage Patterns

### **Current Usage (Warehouse Layer Direct Queries)**

Until gold tables are built, query warehouse layer directly:

```sql
-- Company hiring activity (ready for gold aggregation)
SELECT 
  c.company_name,
  COUNT(DISTINCT f.job_sk) as active_jobs,
  COUNT(*) as total_events,
  MIN(f.event_date) as first_seen,
  MAX(f.event_date) as last_seen
FROM workspace.warehouse.fact_job_postings f
INNER JOIN workspace.warehouse.dim_company c ON f.company_sk = c.company_sk
GROUP BY c.company_name
ORDER BY active_jobs DESC
LIMIT 20;

-- Pipeline health (ready for gold aggregation)
SELECT 
  pipeline_name,
  DATE(started_at) as run_date,
  COUNT(*) as runs,
  AVG(duration_minutes) as avg_duration,
  SUM(CASE WHEN run_status = 'SUCCESS' THEN 1 ELSE 0 END) as successes
FROM workspace.warehouse.fact_pipeline_runs
GROUP BY pipeline_name, DATE(started_at)
ORDER BY run_date DESC, pipeline_name;
```

### **Planned Query Patterns (After Gold Tables Built)**

**Time-Series Analysis** (once `gold_salary_trends` built):

```sql
-- Example: Salary trends over time for Tech sector
SELECT 
  TO_DATE(CAST(salary_date_sk AS STRING), 'yyyyMMdd') AS trend_date,
  salary_max_median,
  delta_vs_prev_month,
  pct_change_vs_prev_month
FROM workspace.gold.gold_salary_trends gst
JOIN workspace.warehouse.dim_sector s ON gst.sector_sk = s.sector_sk
WHERE s.sector_name = 'Technology'
  AND gst.role_sk IS NULL  -- Sector-level rollup
  AND gst.salary_date_sk >= 20230101
ORDER BY trend_date DESC;
```

**Drill-Down Analysis** (once multi-grain tables built):

```sql
-- Example: Sector → Role → Location drill-down
-- Level 1: Sector totals
SELECT sector_sk, SUM(observation_count) AS total_obs
FROM workspace.gold.gold_salary_trends
WHERE role_sk IS NULL AND location_sk IS NULL
GROUP BY sector_sk;

-- Level 2: Drill into specific roles within sector
SELECT role_sk, SUM(observation_count) AS total_obs
FROM workspace.gold.gold_salary_trends
WHERE sector_sk = 5 AND location_sk IS NULL
GROUP BY role_sk;
```

### **Planned Refresh Strategy**

Once gold tables are implemented, recommended refresh cadence:

* **Real-time (Streaming)** - Consider for operational monitoring (`gold_pipeline_health`)
* **Hourly** - High-priority dashboards (hiring trends, company activity)
* **Daily** - Most trend tables (salary, location, sector overview)
* **Weekly** - Aggregated summaries, specialized analytics
* **Ad-hoc** - Backfills, one-off analyses

**Implementation Approach**:
1. Start with daily batch jobs (orchestrated via Databricks Jobs)
2. Use incremental processing where possible (filter by `created_at` or `batch_id`)
3. Materialize common date ranges (last 90 days) fully; archive older data

**Required Metadata** (every gold table):
* `created_at` TIMESTAMP - Last refresh timestamp
* `batch_id` STRING - UUID linking to orchestration run (from `fact_pipeline_runs`)
* Optional: `data_version` INT - Incrementing version number for change tracking

---

## 🔄 Data Flow & Dependencies

### **Upstream Dependencies**

```
Bronze (API Snapshots)
  ↓
Silver (Deduped Jobs)
  ↓
Semantic (Enrichments: Skills, Roles, Companies)
  ↓
Warehouse (Star Schema: Dims + Facts)
  ↓
Gold (Pre-Aggregated Marts) ← YOU ARE HERE
```

### **Primary Source Tables** (Current Status)

**✅ Fully Operational:**
* `workspace.warehouse.fact_job_postings` - 562 posting events tracked (484 unique jobs)
* `workspace.warehouse.fact_pipeline_runs` - 213 pipeline executions logged
* `workspace.warehouse.dim_company` - Company dimension (complete)
* `workspace.warehouse.dim_location` - Location dimension (complete)
* `workspace.warehouse.dim_sector` - Sector dimension (complete)
* `workspace.warehouse.dim_source` - Source dimension (complete)
* `workspace.warehouse.dim_date` - Date dimension (complete)
* `workspace.warehouse.dim_job` - Job SCD2 dimension (484 current versions)

**⚠️ Partial/Mock Data:**
* `workspace.warehouse.fact_salary` - 484 records with mock salary data (awaiting real extraction)
* `workspace.warehouse.dim_role` - Dimension structure ready, but limited mapping coverage

**🚧 Not Yet Implemented:**
* `workspace.warehouse.bridge_job_skill` - Structure exists but not populated
* `workspace.warehouse.dim_skill` - Skill taxonomy not yet defined

### **Table Dependencies & Readiness**

| Gold Table | Primary Warehouse Source | Status |
|------------|-------------------------|--------|
| `gold_pipeline_health` | `fact_pipeline_runs` | ✅ **Ready** (213 runs available) |
| `gold_company_hiring` | `fact_job_postings` + `dim_company` | ✅ **Ready** (562 events, 484 jobs) |
| `gold_location_trends` | `fact_job_postings` + `dim_location` | ✅ **Ready** |
| `gold_hiring_trends` | `fact_job_postings` | ✅ **Ready** (limited role breakdowns) |
| `gold_sector_overview` | `fact_job_postings` + `dim_sector` | ⚠️ **Partial** (posting counts only) |
| `gold_salary_trends` | `fact_salary` | ⚠️ **Blocked** (mock data only) |
| `gold_skill_demand` | `bridge_job_skill` | ⚠️ **Blocked** (bridge empty) |
| `gold_hospitality_*` | Filtered views + subsector taxonomy | 🔜 **Future** |

---

## ⚠️ Known Blockers & Workarounds

### **Blocker 1: Incomplete Role Mapping**

**Impact**: Role-level breakdowns in gold tables will be incomplete  
**Root Cause**: `workspace.intermediate.inter_job_role_map` had plain-text role names instead of structured taxonomy IDs  
**Current State**: Mapping updated but not exhaustive; ~0% role FK resolution in fact tables  
**Workaround**: Build gold tables at sector/company/location grain; omit role dimension until mapping complete  
**Resolution Path**: Expand `inter_job_role_map` with comprehensive title → taxonomy mapping

### **Blocker 2: Missing Salary Data**

**Impact**: `gold_salary_trends` cannot be built with real data  
**Root Cause**: Silver layer (`silver_jobs_current`) lacks salary columns; data exists in bronze JSON but not extracted  
**Current State**: `fact_salary` populated with mock data (484 records: $70K-$100K ranges)  
**Workaround**: Skip salary-dependent gold tables; focus on posting volume/velocity metrics  
**Resolution Path**:  
  1. Parse `bronze_job_snapshot.payload_json` to extract salary fields  
  2. Add columns to silver layer: `salary_min`, `salary_max`, `salary_currency`, `employment_type`  
  3. Re-run `wh_fact_salary` notebook with real data

### **Blocker 3: Skills Extraction Not Implemented**

**Impact**: `gold_skill_demand` and skill-based analytics blocked  
**Root Cause**: Skill extraction from job descriptions not yet implemented; `bridge_job_skill` empty  
**Current State**: Bridge table structure exists but no data  
**Workaround**: Defer skills analytics to later phase  
**Resolution Path**:  
  1. Define skill taxonomy (manual list or ML-extracted)  
  2. Implement NLP extraction from `description_raw` field  
  3. Populate `bridge_job_skill` with extracted skills  
  4. Build gold skill demand tables

### **Recommended Build Order**

**Phase 1 (Unblocked)**:  
1. `gold_pipeline_health` - Operational monitoring  
2. `gold_company_hiring` - Company-level posting trends  
3. `gold_location_trends` - Geographic hiring patterns  
4. `gold_hiring_trends` - Posting velocity (sector-level only)

**Phase 2 (After Upstream Fixes)**:  
5. `gold_sector_overview` - Add salary/skills once available  
6. `gold_salary_trends` - Once real salary data extracted  
7. `gold_skill_demand` - Once skills extraction implemented

**Phase 3 (Future)**:  
8. Hospitality-specific marts  
9. Predictive/ML-based analytics

---

## 📐 Schema Conventions

### **Standard Columns**

All Gold tables include:

| Column | Type | Purpose |
|--------|------|--------|
| `*_date_sk` | INT | Date surrogate key (yyyyMMdd format) |
| `*_sk` | BIGINT | Foreign keys to warehouse dimensions |
| `observation_count` | BIGINT | Number of raw records aggregated |
| `created_at` | TIMESTAMP | Table refresh timestamp |
| `batch_id` | STRING | UUID linking to orchestration run |

### **Temporal Columns**

Trend tables include:
* `prev_month_*` - Prior month comparison value
* `prev_quarter_*` - Prior quarter comparison value
* `delta_vs_prev_month` - Absolute change from previous month
* `pct_change_vs_prev_month` - Percentage change from previous month

### **Rollup Indicators**

NULL dimension keys indicate rollup levels:
* `sector_sk = NULL, role_sk = NULL` → Total across all sectors and roles
* `sector_sk = 5, role_sk = NULL` → Sector 5 total (all roles)
* `sector_sk = -1` → Placeholder for "all sectors" in role-level rollups

### **Naming Conventions**

* **Table Names**: `gold_<domain>_<aggregation>` (e.g., `gold_salary_trends`)
* **Measures**: Descriptive names (e.g., `salary_max_median`, `posting_velocity`)
* **Dimensions**: Foreign key suffix `_sk` (e.g., `sector_sk`, `role_sk`)

---

## ✅ Best Practices

### **When Creating New Gold Tables**

1. **Start with Business Question** - Design aggregations to answer specific BI questions
2. **Pre-Compute Expensive Operations** - Window functions, percentiles, complex joins
3. **Include Multiple Grain Levels** - Enable drill-down without reprocessing
4. **Add Temporal Metrics** - Period-over-period comparisons for trending
5. **Document Minimum Sample Sizes** - Filter for statistical validity (e.g., `observation_count >= 5`)
6. **Test Query Performance** - Gold tables should return in < 2 seconds for dashboards

### **Optimization Techniques**

* **Partition by Date** - Most queries filter on date ranges
* **Z-Order on Common Filters** - Sector, location, company keys
* **Compact Delta Tables** - Regularly OPTIMIZE to merge small files
* **Use DECIMAL for Currency** - Avoid floating-point precision issues
* **Materialize Slowly** - Don't recompute historical data on every run

### **Quality Checks**

* **Row Count Validation** - Compare to warehouse layer (Gold should be <= Warehouse)
* **Null Checks** - Key metrics should never be NULL (use 0 or filter)
* **Temporal Continuity** - Verify no missing dates in time-series
* **Rollup Consistency** - Sector totals should match sum of role-level data

---

## 🔧 Maintenance & Operations

### **Routine Maintenance**

```sql
-- Optimize Gold tables (run weekly)
OPTIMIZE workspace.gold.gold_salary_trends ZORDER BY (sector_sk, role_sk);
OPTIMIZE workspace.gold.gold_skill_demand ZORDER BY (skill_sk, sector_sk);

-- Vacuum old versions (run monthly, retain 30 days)
VACUUM workspace.gold.gold_salary_trends RETAIN 720 HOURS;
```

### **Monitoring Queries**

```sql
-- Check data freshness
SELECT 
  'gold_salary_trends' AS table_name,
  MAX(created_at) AS last_refresh,
  DATEDIFF(HOUR, MAX(created_at), CURRENT_TIMESTAMP()) AS hours_since_refresh
FROM workspace.gold.gold_salary_trends
UNION ALL
SELECT 
  'gold_hiring_trends',
  MAX(created_at),
  DATEDIFF(HOUR, MAX(created_at), CURRENT_TIMESTAMP())
FROM workspace.gold.gold_hiring_trends;

-- Check row counts and growth
SELECT 
  'gold_salary_trends' AS table_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT batch_id) AS unique_batches,
  MIN(salary_date_sk) AS earliest_date,
  MAX(salary_date_sk) AS latest_date
FROM workspace.gold.gold_salary_trends;
```

### **Troubleshooting**

**Issue**: Dashboard queries slow  
**Solution**: Check for missing Z-ordering, run OPTIMIZE

**Issue**: Missing dates in time-series  
**Solution**: Verify warehouse layer has continuous dates, check upstream pipeline failures

**Issue**: Rollup totals don't match detail rows  
**Solution**: Review NULL key conventions, verify aggregation logic

---

## 📚 Related Documentation

* **Warehouse Layer** - See `notebooks/warehouse/README_WAREHOUSE.md` for upstream star schema documentation
* **intermediate layer** - See `notebooks/semantic/README_SEMANTIC.md` for enrichment logic
* **Data Dictionary** - See `docs/data_dictionary.md` for column definitions
* **Dashboard Guides** - See `dashboards/README.md` for BI tool integration

---

## 🏁 Quick Start

### **Current State: Design & Planning Phase**

Gold layer tables are not yet built. To begin development:

1. **Check Warehouse Layer Readiness**:
   ```sql
   -- Verify available data for gold aggregations
   SELECT COUNT(*) FROM workspace.warehouse.fact_job_postings;  -- Should return 562
   SELECT COUNT(*) FROM workspace.warehouse.fact_pipeline_runs; -- Should return 213
   SELECT COUNT(*) FROM workspace.warehouse.dim_company;        -- Active companies
   ```

2. **Build First Gold Table** (Recommended: `gold_pipeline_health`):
   ```python
   # Create notebook: /Users/<your-email>/LMIP/notebooks/gold/gold_pipeline_health
   # Aggregate fact_pipeline_runs by date/pipeline
   # Include: success rate, avg duration, data freshness metrics
   ```

3. **Validate Gold Output**:
   ```sql
   -- After building your first gold table
   SELECT 
     COUNT(*) as total_records,
     COUNT(DISTINCT batch_id) as unique_batches,
     MIN(created_at) as first_load,
     MAX(created_at) as last_load
   FROM workspace.gold.gold_pipeline_health;
   ```

4. **Monitor Upstream Blockers**:
   ```sql
   -- Check role mapping coverage (blocker for role-level analytics)
   SELECT 
     COUNT(*) as total_jobs,
     SUM(CASE WHEN role_sk != -1 THEN 1 ELSE 0 END) as mapped_roles,
     ROUND(100.0 * SUM(CASE WHEN role_sk != -1 THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_mapped
   FROM workspace.warehouse.fact_job_postings;
   
   -- Check if salary data is real or mock
   SELECT DISTINCT salary_min, salary_max, salary_observation_type
   FROM workspace.warehouse.fact_salary
   LIMIT 5;
   -- If observation_type = 'INFERRED' and all values are round numbers, it's mock data
   ```

### **Next Steps for Gold Layer Development**

1. ✅ Build `gold_pipeline_health` first (no blockers)
2. ✅ Build `gold_company_hiring` (posting volume by company)
3. ✅ Build `gold_location_trends` (geographic patterns)
4. ⚠️ Monitor upstream: wait for salary extraction before building `gold_salary_trends`
5. ⚠️ Monitor upstream: wait for skills extraction before building `gold_skill_demand`

---

**Last Updated**: 2026-06-09  
**Current Status**: Warehouse layer stabilization complete; gold layer design finalized  
**Maintained By**: Data Platform Team  
**Questions?**: Contact #data-platform on Slack